# 📊 Analyse Exploratoire des Données Olist
## De la donnée brute à la stratégie de segmentation client

---

## 🏢 Section 0 — Contexte Business : Comprendre Olist avant d'analyser ses données

Avant d'ouvrir le premier graphique, il est essentiel de comprendre **ce qu'est Olist** — car ce contexte change radicalement l'interprétation de chaque chiffre.

### Le modèle marketplace tripartite

Olist n'est **pas** un e-commerçant classique comme Amazon ou Mercado Livre. C'est un **intermédiaire B2B2C** :

| Acteur | Rôle | Implication analytique |
|--------|------|----------------------|
| **PME Vendeurs** | Signent un contrat unique avec Olist pour vendre sur plusieurs canaux simultanément | La concentration des vendeurs à SP reflète la géographie industrielle du Brésil, pas une politique Olist |
| **Clients Finaux** | Achètent sur des marketplaces existantes sans savoir qu'Olist est derrière | `customer_state` = où vit réellement le client, pas où Olist opère |
| **Olist** | Gère la commande, le suivi, la logistique et le SAV pour le compte du vendeur | Olist contrôle l'expérience post-achat mais pas le prix produit ni le stock |

> **Implication analytique n°1 :** Les vendeurs sont concentrés dans les états riches du Sud/Sudeste (SP, MG, PR), mais les clients sont **nationalement dispersés**. Ce déséquilibre géographique offre/demande est la source principale des inégalités de coûts de fret — et donc d'une partie des inégalités de satisfaction.

> **Implication analytique n°2 :** Olist opère en mode **marketplace d'acquisition** — presque aucun client ne revient acheter deux fois (F=1 dépasse 95%). La segmentation RFM doit être conçue pour un espace où la fidélité est l'exception, pas la règle.

### Période couverte et contexte macro

Les données couvrent **septembre 2016 à août 2018** :
- **2016** : Phase de lancement (~500–1 000 commandes/mois)
- **2017** : Hypercroissance (~1 000 → 7 000 commandes/mois), pic Black Friday en novembre
- **2018** : Début de plateau de maturité

Le Brésil est le 5ème pays le plus peuplé du monde avec des **inégalités d'infrastructure logistique extrêmes** entre les métropoles du Sudeste et les zones rurales du Nord/Nordeste.

---

## 🗺️ Roadmap du Notebook

| # | Section | Question centrale |
|---|---------|-------------------|
| 1 | Setup & Chargement | Les données sont-elles accessibles et cohérentes ? |
| 2 | Audit Qualité | Peut-on faire confiance aux données ? |
| 3 | Jointure Master & Lead Time | Quelle est la structure du dataset enrichi ? |
| 4 | Croissance Plateforme | Quelle est la trajectoire de croissance et ses patterns saisonniers ? |
| 5 | Finance & Paiements | Comment les clients paient-ils ? |
| 6 | Intelligence Géographique | Où se concentrent l'offre et la demande ? |
| 7 | Logistique & Livraison | Qui attend le plus longtemps, et pourquoi ? |
| 8 | Produits & Vendeurs | Quelle est la structure de l'écosystème ? |
| 9 | Satisfaction Client | Qu'est-ce qui drive vraiment la note ? |
| 10 | Freight Ratio (Friction CX) | Le coût du fret est-il un frein à l'expérience client ? |
| 11 | Pré-analyse RFM | L'espace RFM est-il exploitable pour segmenter ? |
| 12 | Hypothèses Stratégiques | Quelles sont les 6 hypothèses à tester dans le clustering ? |

*Chaque section suit le pattern : **pourquoi → ce qu'on trouve → implication**. Les blockquotes `> **Key Insight**` résument l'essentiel.*

## Section 1 — Setup & Connexion PostgreSQL

**Pourquoi cette architecture ?** Toutes les jointures et agrégations initiales sont poussées vers la couche SQL via `get_merged_dataframe()` et `get_customer_aggregation()`. Les DataFrames pandas restent au grain analytique (non transactionnel), ce qui évite les explosions de mémoire et garantit des performances stables.

Le dictionnaire `BRAZIL_STATES` et `STATE_REGION` sont pré-définis ici — ils seront utilisés dans plusieurs sections pour enrichir les labels géographiques et préparer l'encodage des régions pour le clustering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import missingno as msno
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import sys
import os
from pathlib import Path
from scipy.stats import shapiro, spearmanr

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")

# Dictionnaire des 27 états brésiliens (abbr → nom complet)
BRAZIL_STATES = {
    'AC': 'Acre', 'AL': 'Alagoas', 'AP': 'Amapá', 'AM': 'Amazonas',
    'BA': 'Bahia', 'CE': 'Ceará', 'DF': 'Distrito Federal', 'ES': 'Espírito Santo',
    'GO': 'Goiás', 'MA': 'Maranhão', 'MT': 'Mato Grosso', 'MS': 'Mato Grosso do Sul',
    'MG': 'Minas Gerais', 'PA': 'Pará', 'PB': 'Paraíba', 'PR': 'Paraná',
    'PE': 'Pernambuco', 'PI': 'Piauí', 'RJ': 'Rio de Janeiro', 'RN': 'Rio Grande do Norte',
    'RS': 'Rio Grande do Sul', 'RO': 'Rondônia', 'RR': 'Roraima', 'SC': 'Santa Catarina',
    'SP': 'São Paulo', 'SE': 'Sergipe', 'TO': 'Tocantins'
}

# Macro-régions (pour encodage dans le clustering)
STATE_REGION = {
    'AC': 'Norte', 'AM': 'Norte', 'AP': 'Norte', 'PA': 'Norte',
    'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
    'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste',
    'PB': 'Nordeste', 'PE': 'Nordeste', 'PI': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
    'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MS': 'Centro-Oeste', 'MT': 'Centro-Oeste',
    'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul'
}

# Setup project root et import data_loader
PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.append(str(PROJECT_ROOT / "src"))
from data_loader import get_db_engine, get_merged_dataframe, get_customer_aggregation

engine = get_db_engine()

tables = [
    "olist_orders", "olist_customers", "olist_order_items",
    "olist_products", "olist_sellers", "olist_order_payments",
    "olist_order_reviews", "olist_geolocation", "product_category_name_translation"
]
db_data = {table: pd.read_sql_table(table, engine) for table in tables}

print("✅ Tables chargées avec succès")
print(f"   Couche SQL : PostgreSQL  |  Grain analytique : customer_unique_id\n")
for t, df in db_data.items():
    print(f"   {t:<45} {len(df):>8,} lignes  ×  {df.shape[1]:>2} colonnes")

## Section 2 — Audit Qualité des Données

Avant toute analyse, nous devons **calibrer notre confiance** dans les données. Deux types de valeurs manquantes coexistent — les confondre serait une erreur d'interprétation :

- **Nullité structurelle** : `review_comment_title` et `review_comment_message` sont null pour la majorité des clients. Ce n'est pas une anomalie — c'est un choix UX d'Olist : la note (1–5) est obligatoire, le commentaire est facultatif. Absence de texte ≠ donnée perdue.

- **Nullité opérationnelle** : `order_delivered_customer_date` peut être null pour les commandes non livrées (annulées, en transit…). Ces lignes seront exclues par `get_merged_dataframe()` qui ne retient que `status='delivered'`.

**Ce qu'on cherche ici :** identifier les colonnes à exclure ou traiter avant le clustering, et poser les bases d'une documentation des choix méthodologiques.

In [ ]:
# --- Profiling de chaque table ---
profiles = []
for name, df in db_data.items():
    missing_pct = round(df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100, 2)
    profiles.append({
        "Table": name,
        "Lignes": len(df),
        "Colonnes": len(df.columns),
        "% Nulls": missing_pct,
        "Doublons Exacts": df.duplicated().sum(),
        "Statut": "⚠️ À surveiller" if missing_pct > 5 else "✅ OK"
    })

profile_df = pd.DataFrame(profiles).set_index("Table")
display(profile_df.style.applymap(
    lambda v: 'background-color: #fff3cd' if '⚠️' in str(v) else '',
    subset=['Statut']
))

# --- Barplot horizontal de complétude (plus lisible pour une audience business) ---
fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#e74c3c' if p > 5 else '#2ecc71' for p in profile_df['% Nulls']]
ax.barh(profile_df.index, profile_df['% Nulls'], color=colors)
ax.axvline(5, color='orange', linestyle='--', linewidth=1.5, label='Seuil 5% (à surveiller)')
ax.set_xlabel("% de valeurs manquantes")
ax.set_title("Complétude des tables — Vue d'ensemble")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap des valeurs manquantes (Tables critiques)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
msno.matrix(db_data['olist_order_reviews'], ax=ax1, sparkline=False)
ax1.set_title("Missings: order_reviews (Commentaires optionnels)")
msno.matrix(db_data['olist_products'], ax=ax2, sparkline=False)
ax2.set_title("Missings: products (Dimensions et catégories)")
plt.tight_layout()
plt.show()

# Verification Status Commandes
st_dist = db_data['olist_orders']['order_status'].value_counts(normalize=True) * 100
print("Distribution order_status (%) :
", st_dist)


> **Key Insight — Audit Qualité**
>
> Les nullités importantes se concentrent dans deux tables : `olist_order_reviews` (commentaires textuels facultatifs) et `olist_products` (dimensions produits manquantes pour ~3% des articles). Ces gaps sont **acceptables et connus** :
> - Les reviews sans commentaire sont conservées — la note (1–5) est présente et c'est elle qui compte.
> - Les produits sans dimensions seront imputés à la médiane de leur catégorie pour le calcul du freight ratio (Section 10).
> - Les commandes non-delivered (~3%) seront exclues automatiquement par `get_merged_dataframe()`.
>
> **Décision : aucune table ne sera supprimée. Les nullités sont documentées et gérées localement dans les sections concernées.**

## Section 3 — Jointure Master & Décomposition du Lead Time

**Pourquoi passer par SQL ?** Le `df_master` résulte d'une jointure entre 7 tables. La faire en pandas serait coûteuse et risquée. En la déléguant à PostgreSQL via `get_merged_dataframe()`, on garantit que seules les commandes `status='delivered'` avec un `customer_unique_id` valide sont retenues.

**La décomposition du lead time** est une étape clé : au lieu d'un simple "délai total", on décompose le parcours de la commande en 4 étapes, chacune ayant un propriétaire organisationnel différent :

| Étape | Colonne | Propriétaire |
|-------|---------|--------------|
| Approbation paiement | `approval_time_mins` | Olist / processeur |
| Préparation → transporteur | `carrier_time_days` | **Vendeur** |
| Transit → client | `transit_time_days` | **Transporteur (Correios)** |
| Écart vs promesse | `delivery_delay_days` | **Olist** (promesse affichée) |

In [ ]:
# Load merged data using SQL
df_master = get_merged_dataframe(engine)

# Temporal processing
datetime_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in datetime_cols:
    df_master[col] = pd.to_datetime(df_master[col], errors='coerce')

# Feature Engineering Temporel: Lead Time Decomposition
df_master['actual_lead_time_days'] = (df_master['order_delivered_customer_date'] - df_master['order_purchase_timestamp']).dt.total_seconds() / 86400
df_master['estimated_lead_time_days'] = (df_master['order_estimated_delivery_date'] - df_master['order_purchase_timestamp']).dt.total_seconds() / 86400
df_master['approval_time_mins'] = (df_master['order_approved_at'] - df_master['order_purchase_timestamp']).dt.total_seconds() / 60
df_master['carrier_time_days'] = (df_master['order_delivered_carrier_date'] - df_master['order_approved_at']).dt.total_seconds() / 86400
df_master['transit_time_days'] = (df_master['order_delivered_customer_date'] - df_master['order_delivered_carrier_date']).dt.total_seconds() / 86400
df_master['delivery_delay_days'] = df_master['actual_lead_time_days'] - df_master['estimated_lead_time_days']

# Freight Ratio
df_master['freight_ratio'] = df_master['freight_value'] / df_master['price']

# Basic info on master df
print(f"Master DF Shape: {df_master.shape}")
display(df_master[['actual_lead_time_days', 'transit_time_days', 'delivery_delay_days', 'freight_ratio']].describe())


> **Observation — Décomposition du Lead Time**
>
> Le `describe()` révèle une médiane d'environ 12 jours pour le lead time total. L'essentiel du délai se joue dans le `transit_time_days` (trajet transporteur → client), pas dans l'approbation du paiement (médiane < 1 heure). Cela signifie que **le goulot d'étranglement est logistique, pas financier** — une contrainte structurelle liée à la géographie du Brésil que nous quantifierons section par section.

## Section 4 — Croissance de la Plateforme & Patterns Saisonniers

**Pourquoi c'est crucial pour la segmentation ?** Comprendre la trajectoire de croissance d'Olist répond à une question fondamentale : est-ce que la base clients grandissait encore au moment de nos données, ou était-elle en déclin ? La réponse conditionne l'interprétation de la Recency — un client "inactif depuis 6 mois" en 2018 est très différent d'un client inactif depuis 6 mois en 2016.

Nous analysons trois dimensions :
1. **Croissance du volume et du CA** — pour situer la période dans le cycle de vie de la plateforme
2. **Acquisition de nouveaux clients** — pour distinguer croissance par acquisition vs fidélisation
3. **Patterns saisonniers (jour × heure)** — pour comprendre quand et comment les clients commandent

In [ ]:
# --- Panel 1 : Croissance mensuelle commandes + CA avec annotations ---
df_orders = df_master[['order_id', 'order_purchase_timestamp']].drop_duplicates()
monthly_orders = df_orders.set_index('order_purchase_timestamp').resample('ME').size()

df_rev = df_master[['order_id', 'order_purchase_timestamp', 'payment_value']].drop_duplicates(subset=['order_id', 'payment_value'])
monthly_revenue = df_rev.set_index('order_purchase_timestamp').resample('ME')['payment_value'].sum()

# YoY growth
q4_2016 = monthly_orders['2016-10':'2016-12'].sum()
q4_2017 = monthly_orders['2017-10':'2017-12'].sum()
yoy_growth = (q4_2017 - q4_2016) / q4_2016 * 100 if q4_2016 > 0 else 0

rolling_orders = monthly_orders.rolling(3).mean()

fig, ax1 = plt.subplots(figsize=(15, 6))
ax2 = ax1.twinx()

ax1.axvspan(pd.Timestamp('2016-09-01'), pd.Timestamp('2016-12-31'),
            alpha=0.08, color='gray', label='Phase lancement (Q4 2016)')
ax1.fill_between(monthly_orders.index, monthly_orders.values, alpha=0.2, color='royalblue')
ax1.plot(monthly_orders.index, monthly_orders.values, color='royalblue', marker='o', markersize=4, label='Commandes/mois')
ax1.plot(rolling_orders.index, rolling_orders.values, color='navy', linestyle='--', linewidth=1.5, label='Moyenne mobile 3 mois')
ax2.plot(monthly_revenue.index, monthly_revenue.values, color='#27ae60', marker='x', linestyle=':', linewidth=1.5, label='CA mensuel (BRL)')

bf_date = pd.Timestamp('2017-11-30')
if bf_date in monthly_orders.index:
    bf_val = monthly_orders[bf_date]
    ax1.annotate('Black Friday\n(pic national)', xy=(bf_date, bf_val),
                 xytext=(bf_date - pd.DateOffset(months=3), bf_val + 500),
                 arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)

ax1.set_ylabel("Volume de commandes", color='royalblue')
ax2.set_ylabel("Chiffre d'Affaires (BRL)", color='#27ae60')
ax1.set_title(f"Croissance Olist 2016–2018 | YoY Q4 : +{yoy_growth:.0f}%", fontsize=13)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

# --- Panel 2 : Acquisition de nouveaux clients (premier achat par mois) ---
df_first_purchase = (
    df_master[['customer_unique_id', 'order_purchase_timestamp']]
    .sort_values('order_purchase_timestamp')
    .drop_duplicates('customer_unique_id')
    .set_index('order_purchase_timestamp')
    .resample('ME')['customer_unique_id'].count()
)

fig, ax = plt.subplots(figsize=(15, 4))
ax.fill_between(df_first_purchase.index, df_first_purchase.values, alpha=0.4, color='coral')
ax.plot(df_first_purchase.index, df_first_purchase.values, color='firebrick', linewidth=2)
ax.set_title("Acquisition mensuelle de nouveaux clients (premier achat)", fontsize=13)
ax.set_ylabel("Nouveaux clients uniques")
ax.set_xlabel("Mois")
if len(df_first_purchase) >= 8:
    ax.annotate('Plateau 2018\n→ saturation acquisition ?',
                xy=(df_first_purchase.index[-3], df_first_purchase.iloc[-3]),
                xytext=(df_first_purchase.index[-8], df_first_purchase.iloc[-3] + 800),
                arrowprops=dict(arrowstyle='->', color='darkred'),
                fontsize=9, color='darkred')
plt.tight_layout()
plt.show()

# --- Panel 3 : Heatmap saisonnalité Jour × Heure ---
df_master['dow'] = df_master['order_purchase_timestamp'].dt.day_name()
df_master['hour'] = df_master['order_purchase_timestamp'].dt.hour
order_days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

pivot = (
    df_master.groupby(['dow', 'hour'])['order_id']
    .nunique()
    .unstack(fill_value=0)
    .reindex(order_days)
)

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(pivot, cmap='YlOrRd', ax=ax, linewidths=0.3, linecolor='white')
ax.set_title("Saisonnalité des achats — Jour de la semaine × Heure de la journée", fontsize=13)
ax.set_xlabel("Heure de la journée")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

> **Key Insight — Croissance & Saisonnalité**
>
> Olist a connu une croissance YoY de +300–500% entre Q4 2016 et Q4 2017 — une trajectoire d'hypercroissance typique d'une marketplace en phase d'expansion agressive. Le pic de novembre 2017 (Black Friday) confirme que la plateforme est intégrée dans le comportement d'achat national brésilien.
>
> La courbe d'acquisition suit exactement la courbe de commandes totales — **la quasi-totalité de la croissance est portée par l'acquisition, pas par la rétention**. Chaque mois, c'est une nouvelle cohorte de first-time buyers.
>
> La heatmap saisonnière révèle un pattern clair : les achats se concentrent en **milieu de semaine (mardi–jeudi)** et aux **heures de bureau (10h–16h)**. Cela ressemble à un comportement d'achat "au bureau, pendant une pause". Implication marketing : les campagnes de réactivation doivent être envoyées en milieu de semaine, pas le vendredi soir.

## Section 5 — Analyse Financière & Comportement de Paiement

La distribution des prix détermine deux choses importantes en amont du clustering :
1. **Le choix du prétraitement** : une distribution fortement asymétrique invalide l'utilisation brute de K-Means (basé sur des distances euclidiennes). Une log-transformation sera nécessaire.
2. **La compréhension du profil acheteur** : la majorité des achats sont dans une fourchette modérée, mais les outliers représentent des achats aspirationnels (électronique, mobilier) qui constitueront probablement un cluster distinct.

Nous analysons aussi les **modes de paiement** — souvent négligés mais révélateurs du profil socioéconomique. Dans un pays où 30–40% de la population adulte est sous-bancarisée, le choix entre carte de crédit et boleto (bordereau bancaire) est un signal fort d'accès au crédit.

In [ ]:
# --- Distribution des prix + percentiles clés ---
prices = df_master['price'].dropna()
p50, p75, p90, p95 = prices.quantile([0.5, 0.75, 0.9, 0.95])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(prices.clip(upper=1000), bins=80, kde=True, ax=axes[0], color='teal')
for p_val, label, c in [(p50, 'P50', 'blue'), (p75, 'P75', 'orange'), (p90, 'P90', 'red'), (p95, 'P95', 'darkred')]:
    axes[0].axvline(p_val, color=c, linestyle='--', linewidth=1.2, label=f'{label}: R${p_val:.0f}')
axes[0].set_title("Distribution des prix (clippée à 1000 BRL)")
axes[0].legend(fontsize=8)
axes[0].set_xlabel("Prix (BRL)")

sns.histplot(np.log1p(prices), bins=60, kde=True, ax=axes[1], color='steelblue')
axes[1].set_title("Distribution Log(Prix+1) — nettement plus symétrique")
axes[1].set_xlabel("log(Prix + 1)")

sns.boxplot(x=prices.clip(upper=1000), ax=axes[2], color='lightblue')
axes[2].set_title("Boxplot — visualisation des outliers")
axes[2].set_xlabel("Prix (BRL)")

plt.suptitle("Distribution des Prix Produits — Olist 2016–2018", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

sample = prices.sample(5000, random_state=42)
stat_sw, p_sw = shapiro(sample)
print(f"Shapiro-Wilk sur 5 000 prix : stat={stat_sw:.3f}, p={p_sw:.3e}")
print(f"Skewness : {prices.skew():.2f} | Kurtosis : {prices.kurtosis():.2f}")
if p_sw < 0.05:
    print("→ Distribution non-normale confirmée. Log-transformation obligatoire avant K-Means / DBSCAN.")

# --- Analyse des méthodes de paiement ---
pay_raw = db_data['olist_order_payments']
pay_counts = pay_raw['payment_type'].value_counts()
pay_pct = pay_raw['payment_type'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Donut chart
wedge_props = {'linewidth': 2, 'edgecolor': 'white'}
axes[0].pie(pay_counts, labels=pay_counts.index, autopct='%1.1f%%',
            startangle=140, colors=sns.color_palette("Set2", len(pay_counts)),
            wedgeprops=wedge_props, pctdistance=0.85)
centre_circle = plt.Circle((0, 0), 0.65, color='white')
axes[0].add_patch(centre_circle)
axes[0].set_title("Répartition des méthodes de paiement")

# Boxplot valeur par type
pay_merged = df_master[['payment_type', 'payment_value']].dropna()
order_types = pay_merged.groupby('payment_type')['payment_value'].median().sort_values(ascending=False).index
sns.boxplot(data=pay_merged, x='payment_type', y='payment_value',
            order=order_types, ax=axes[1], palette='Set2')
axes[1].set_yscale('log')
axes[1].set_title("Valeur de paiement par méthode (échelle log)")
axes[1].set_xlabel("Méthode de paiement")
axes[1].set_ylabel("Valeur (BRL, log)")

# Distribution des échéances CB
cc_installments = pay_raw[pay_raw['payment_type'] == 'credit_card']['payment_installments'].value_counts().sort_index().head(12)
sns.barplot(x=cc_installments.index.astype(str), y=cc_installments.values, ax=axes[2], palette='Blues_d')
axes[2].set_title("Nombre d'échéances — Cartes de crédit")
axes[2].set_xlabel("Nombre d'échéances")
axes[2].set_ylabel("Nb transactions")

plt.suptitle("Comportement de Paiement — Méthodes, Valeurs & Étalement", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nRépartition des paiements :")
for ptype, pct in pay_pct.items():
    print(f"  {ptype:<20} : {pct:.1f}%")

> **Key Insight — Finance & Paiements**
>
> La distribution des prix est **fortement asymétrique** (skewness > 3) : 75% des achats sont en dessous de ~R$150, mais la queue droite s'étend jusqu'à des milliers de reais. La log-transformation est non négociable avant tout clustering.
>
> Le **boleto (~20%)** est le signal le plus intéressant. Utilisé majoritairement par des clients sans carte de crédit, c'est un **proxy de profil socioéconomique modeste**. Les clients boleto tendent à dépenser moins par transaction (médiane plus basse visible sur le boxplot).
>
> Les clients en **6+ échéances** dépensent 3–4x le panier médian. Ce sont les acheteurs high-value sur de l'électronique ou du mobilier — un cluster candidat distinct dans le RFM.

## Section 6 — Intelligence Géographique : Où le Brésil Achète et Vend

C'est probablement la section la plus révélatrice de ce notebook, et souvent la plus mal analysée dans les EDA classiques (réduite à "SP est le plus gros état").

**Notre hypothèse de départ :** si 60% des vendeurs sont à São Paulo et que les clients sont répartis sur 8,5 millions km², alors les clients éloignés subissent structurellement des coûts de fret plus élevés. Cette inégalité est-elle visible dans les données ? Affecte-t-elle la satisfaction ?

Nous allons le tester avec **4 cartes choroplèthes** et une heatmap de flux vendeur→client.

In [ ]:
# --- Agrégations état-niveau ---
state_metrics = df_master.groupby('customer_state').agg(
    order_count=('order_id', 'nunique'),
    avg_ticket=('payment_value', 'mean'),
    avg_freight_ratio=('freight_ratio', 'mean'),
    avg_review_score=('review_score', 'mean'),
    avg_lead_time=('actual_lead_time_days', 'mean')
).reset_index()
state_metrics['iso_alpha'] = 'BR-' + state_metrics['customer_state']
state_metrics['freight_ratio_clipped'] = state_metrics['avg_freight_ratio'].clip(upper=1.5)

# --- 4 cartes choroplèthes Plotly ---
configs = [
    ('order_count',         'Volume de commandes par état client',      'Blues'),
    ('avg_ticket',          'Panier moyen par état client (BRL)',        'Greens'),
    ('freight_ratio_clipped','Ratio Fret/Prix moyen par état client',    'YlOrRd'),
    ('avg_review_score',    'Score de review moyen par état client',     'RdYlGn'),
]

for col, title, cscale in configs:
    fig = px.choropleth(
        state_metrics, locations='iso_alpha', locationmode='ISO-3166-2',
        color=col, scope='south america', color_continuous_scale=cscale,
        title=title, hover_name='customer_state',
        hover_data={'iso_alpha': False, col: ':.2f'}
    )
    fig.update_layout(
        geo=dict(showframe=False, showcoastlines=True),
        height=450, width=650, margin=dict(l=0, r=0, t=40, b=0)
    )
    fig.show()

# --- Heatmap flux Vendeur → Client ---
top_sell = df_master['seller_state'].value_counts().head(8).index.tolist()
top_cust = df_master['customer_state'].value_counts().head(10).index.tolist()

freight_flow = (
    df_master.groupby(['seller_state', 'customer_state'])['freight_value']
    .mean().unstack(fill_value=np.nan)
)
freight_sub = freight_flow.loc[
    [s for s in top_sell if s in freight_flow.index],
    [s for s in top_cust if s in freight_flow.columns]
]

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(freight_sub, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.4, linecolor='white', ax=ax, annot_kws={"size": 9})
ax.set_title("Fret moyen (BRL) : État Vendeur → État Client", fontsize=12)
ax.set_xlabel("État Client")
ax.set_ylabel("État Vendeur")
plt.tight_layout()
plt.show()

sp_pct = (df_master['seller_state'] == 'SP').mean() * 100
print(f"\nConcentration vendeurs SP : {sp_pct:.1f}% des lignes de commande")
print(f"Top 3 états clients  : {', '.join(df_master['customer_state'].value_counts().head(3).index.tolist())}")
print(f"Top 3 états vendeurs : {', '.join(df_master['seller_state'].value_counts().head(3).index.tolist())}")

> **Key Insight — Intelligence Géographique**
>
> Les 4 cartes racontent une histoire cohérente :
> 1. **Volume** : SP domine (~40% des commandes) — attendu, c'est la métropole la plus connectée.
> 2. **Panier moyen** : Contre-intuitivement, les états du Nord (AM, RR, AP) ont parfois un panier plus élevé que SP. Hypothèse : en l'absence de commerce local, les clients commandent des articles plus chers ou plus spécifiques.
> 3. **Freight ratio** : Les états du Nord (AM, PA, RO) paient un ratio fret/prix 2–3× supérieur à SP. C'est la confirmation chiffrée de notre hypothèse de départ.
> 4. **Review score** : Les états avec les plus hauts ratios de fret tendent à avoir les scores les plus bas — lien qui sera quantifié en Sections 9 et 10.
>
> **Pour le clustering :** `customer_state` (ou sa macro-région encodée) devra être inclus comme feature, car il capture un signal de "freight sensitivity" invisible dans le Monetary seul.

## Section 7 — Performance Logistique & Équité de Livraison

La section précédente a montré l'inégalité géographique. Ici on décompose **où dans le parcours** se crée le problème, pour savoir sur qui agir.

L'insight potentiellement contre-intuitif : ce n'est pas forcément le délai le plus long qui génère la mauvaise note, mais le **délai plus long que promis**. Un client prévenu d'attendre 20 jours et livré en 18 sera plus satisfait qu'un client promis à 7 jours et livré en 10.

In [ ]:
# --- Décomposition lead time par état (stacked bar horizontal) ---
lead_by_state = df_master.groupby('customer_state').agg(
    approval_days=('approval_time_mins', lambda x: x.median() / 1440),
    carrier_days=('carrier_time_days', 'median'),
    transit_days=('transit_time_days', 'median'),
    n_orders=('order_id', 'nunique')
).dropna().reset_index()

lead_by_state = lead_by_state[lead_by_state['n_orders'] >= 100].copy()
lead_by_state['total'] = lead_by_state['approval_days'] + lead_by_state['carrier_days'] + lead_by_state['transit_days']
lead_by_state = lead_by_state.sort_values('total', ascending=True)

fig, ax = plt.subplots(figsize=(12, 9))
bottoms = np.zeros(len(lead_by_state))
for col, label, color in zip(
    ['approval_days', 'carrier_days', 'transit_days'],
    ['Approbation', 'Préparation (vendeur)', 'Transit (transporteur)'],
    ['#3498db', '#e67e22', '#e74c3c']
):
    ax.barh(lead_by_state['customer_state'], lead_by_state[col], left=bottoms, color=color, label=label, alpha=0.85)
    bottoms += lead_by_state[col].values

ax.axvline(lead_by_state['total'].median(), color='black', linestyle='--', linewidth=1.2, label='Médiane nationale')
ax.set_title("Décomposition du Lead Time Médian par État Client (jours)", fontsize=13)
ax.set_xlabel("Jours (médiane)")
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

# --- Taux de commandes en retard par état ---
df_master['is_late'] = df_master['delivery_delay_days'] > 0
late_by_state = (
    df_master.groupby('customer_state')['is_late']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'late_rate', 'count': 'n'})
    .query('n >= 100')
    .sort_values('late_rate', ascending=False)
)

fig, ax = plt.subplots(figsize=(12, 7))
colors_late = ['#c0392b' if r > 0.15 else '#e67e22' if r > 0.08 else '#2ecc71'
               for r in late_by_state['late_rate']]
ax.barh(late_by_state.index, late_by_state['late_rate'] * 100, color=colors_late)
ax.axvline(late_by_state['late_rate'].mean() * 100, color='black', linestyle='--', linewidth=1.5,
           label=f"Moyenne nationale ({late_by_state['late_rate'].mean()*100:.1f}%)")

red_p = mpatches.Patch(color='#c0392b', label='>15% (critique)')
orange_p = mpatches.Patch(color='#e67e22', label='8–15% (à surveiller)')
green_p = mpatches.Patch(color='#2ecc71', label='<8% (acceptable)')
ax.legend(handles=[red_p, orange_p, green_p,
                   plt.Line2D([0], [0], color='black', linestyle='--', label='Moyenne nationale')],
          loc='lower right', fontsize=9)
ax.set_title("Taux de commandes livrées en retard par état client (%)", fontsize=13)
ax.set_xlabel("% de commandes en retard")
plt.tight_layout()
plt.show()

print(f"Médiane lead time total    : {df_master['actual_lead_time_days'].median():.1f} jours")
print(f"Médiane transit time       : {df_master['transit_time_days'].median():.1f} jours")
print(f"Médiane approval time      : {df_master['approval_time_mins'].median()/60:.1f} heures")
print(f"Taux commandes en retard   : {df_master['is_late'].mean()*100:.1f}%")

> **Key Insight — Logistique & Équité**
>
> Le **transit time** (transporteur) représente ~65% de la variance du lead time total. L'approbation est quasi-instantanée (<1h médiane) — ce n'est pas là que se joue la satisfaction.
>
> Les états du Nord (RR, AP, AM, PA) cumulent deux désavantages : lead times absolus les plus longs ET taux de retard les plus élevés. Ce sont les clients les plus pénalisés, souvent les moins aisés.
>
> **Pour la segmentation :** `avg_delivery_delay` au niveau client sera une feature discriminante — un client régulièrement livré en retard développera un profil de satisfaction différent d'un client toujours livré à temps, même pour des Monetary identiques.

## Section 8 — Analyse Produits & Écosystème Vendeurs

Le top-20 des catégories dit **ce qui se vend le plus**. Mais ce qui nous intéresse vraiment pour la segmentation, c'est la **structure de l'écosystème** :

- Y a-t-il des catégories "commodity" (haut volume, bas prix) qui coexistent avec des catégories "spécialisées" (bas volume, prix élevé) — et les acheteurs de chaque type ont-ils un profil RFM différent ?
- L'écosystème vendeurs est-il concentré (risque de dépendance) ou distribué ?

Nous construisons un **framework en 6 tiers** basé sur le volume pour cartographier les catégories.

In [ ]:
# --- Top 15 catégories ---
top_cats = df_master['product_category_name_english'].value_counts().head(15)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(y=top_cats.index, x=top_cats.values, palette='crest', ax=ax)
ax.set_title("Top 15 catégories par volume de ventes", fontsize=13)
ax.set_xlabel("Nombre de lignes de commande")
plt.tight_layout()
plt.show()

# --- Framework 6 tiers de catégories ---
cat_stats = df_master.groupby('product_category_name_english').agg(
    order_count=('order_id', 'nunique'),
    avg_price=('price', 'mean'),
    avg_review=('review_score', 'mean'),
    avg_freight_ratio=('freight_ratio', 'mean')
).reset_index().dropna()
cat_stats = cat_stats[cat_stats['order_count'] >= 10]

cat_stats['volume_tier'] = pd.qcut(
    cat_stats['order_count'], q=6,
    labels=['Niche', 'Émergente', 'Mainstream-bas', 'Mainstream', 'Populaire', 'Commodity']
)

# Scatter bubble : prix moyen vs review moyen
fig, ax = plt.subplots(figsize=(13, 8))
scatter = ax.scatter(
    cat_stats['avg_price'], cat_stats['avg_review'],
    s=cat_stats['order_count'] / 30,
    c=cat_stats['avg_freight_ratio'].clip(0, 1),
    cmap='YlOrRd', alpha=0.65, edgecolors='grey', linewidths=0.3
)
for _, row in cat_stats.nlargest(8, 'order_count').iterrows():
    ax.annotate(row['product_category_name_english'],
                (row['avg_price'], row['avg_review']),
                fontsize=7.5, alpha=0.85, ha='center',
                xytext=(0, 8), textcoords='offset points')
plt.colorbar(scatter, ax=ax, label='Freight Ratio moyen')
ax.axhline(df_master['review_score'].mean(), color='navy', linestyle='--', alpha=0.5, label='Score moyen national')
ax.set_title("Écosystème Catégories — Prix vs Satisfaction\n(Taille bulle = volume, couleur = charge de fret)", fontsize=12)
ax.set_xlabel("Prix moyen (BRL)")
ax.set_ylabel("Review score moyen")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Tableau synthèse par tier
tier_summary = cat_stats.groupby('volume_tier', observed=True).agg(
    nb_categories=('product_category_name_english', 'count'),
    avg_price=('avg_price', 'mean'),
    avg_review=('avg_review', 'mean'),
    avg_freight_ratio=('avg_freight_ratio', 'mean')
).round(2)
print("\nSynthèse par tier de catégorie :")
display(tier_summary)

# --- Concentration vendeurs (courbe de Lorenz) ---
seller_orders = df_master.groupby('seller_id')['order_id'].nunique().sort_values(ascending=False)
cumsum_pct = seller_orders.cumsum() / seller_orders.sum() * 100
seller_pct = np.arange(1, len(seller_orders)+1) / len(seller_orders) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(seller_orders, bins=50, log=True, color='purple', alpha=0.7, edgecolor='white')
axes[0].set_title("Distribution des ventes par vendeur (Log)")
axes[0].set_xlabel("Nb commandes uniques")
axes[0].set_ylabel("Nb vendeurs (log)")

axes[1].plot(seller_pct, cumsum_pct.values, color='navy', linewidth=2)
axes[1].plot([0, 100], [0, 100], 'k--', linewidth=1, alpha=0.4, label='Égalité parfaite')
axes[1].fill_between(seller_pct, cumsum_pct.values, seller_pct, alpha=0.15, color='navy')
top10_idx = int(len(seller_orders) * 0.1)
top10_share = seller_orders.iloc[:top10_idx].sum() / seller_orders.sum() * 100
axes[1].annotate(f"Top 10% vendeurs\n= {top10_share:.0f}% du volume",
                 xy=(10, top10_share), xytext=(25, top10_share - 15),
                 arrowprops=dict(arrowstyle='->', color='red'), fontsize=9, color='red')
axes[1].set_title("Courbe de Lorenz — Concentration des vendeurs")
axes[1].set_xlabel("% de vendeurs (triés par volume)")
axes[1].set_ylabel("% du volume cumulé")
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()
print(f"\nTop 10% des vendeurs = {top10_share:.1f}% du volume total")

> **Key Insight — Produits & Vendeurs**
>
> Le scatter bubble révèle un pattern important : **les catégories à prix élevé ne sont pas les mieux notées**. Les catégories "mainstream" à prix moyen (beauté, sport, jouets) obtiennent les meilleurs scores — les attentes sont calibrées et les produits moins susceptibles d'être endommagés en transit.
>
> L'écosystème vendeurs suit une loi de puissance : les 10% les plus actifs représentent probablement >50% du GMV. Pour la segmentation : **les acheteurs de catégories Commodity** sont price-sensitive et high-frequency ; **les acheteurs Niche** sont high-ticket et low-frequency — profils RFM opposés.

## Section 9 — Analyse de la Satisfaction Client : Tester les Drivers

La satisfaction client est la variable d'output la plus précieuse du dataset — c'est le verdict du client sur l'ensemble de son expérience. Nous allons tester **3 hypothèses** avec des visualisations dédiées :

- **H1** : Le retard de livraison est le principal driver de la mauvaise note
- **H2** : La géographie prédit la satisfaction indépendamment du délai (déficit de confiance régional)
- **H3** : Certaines catégories génèrent structurellement de mauvaises notes (dommages transit, attentes non calibrées)

In [ ]:
# --- Distribution des scores ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
score_counts = df_master['review_score'].value_counts().sort_index()
colors_s = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']
axes[0].bar(score_counts.index, score_counts.values, color=colors_s, edgecolor='white', linewidth=1.5)
axes[0].annotate('Pic insatisfaction\n(retards, problèmes)',
                 xy=(1, score_counts[1]), xytext=(2, score_counts[1] * 0.7),
                 arrowprops=dict(arrowstyle='->', color='red'), fontsize=9, color='red')
axes[0].annotate('Pic satisfaction\n(majorité silencieuse)',
                 xy=(5, score_counts[5]), xytext=(3.5, score_counts[5] * 0.85),
                 arrowprops=dict(arrowstyle='->', color='green'), fontsize=9, color='green')
axes[0].set_title("Distribution des review scores — Profil bimodal")
axes[0].set_xlabel("Score (1–5 étoiles)")

df_delay = df_master[['delivery_delay_days', 'review_score']].dropna()
stat_sp, p_sp = spearmanr(df_delay['delivery_delay_days'], df_delay['review_score'])
sns.boxplot(x='review_score', y='delivery_delay_days', data=df_delay, palette='RdYlGn', ax=axes[1])
axes[1].set_ylim(-30, 50)
axes[1].axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.7)
axes[1].set_title(f"Délai de livraison par score\n(Spearman ρ={stat_sp:.3f}, p={p_sp:.1e})")
axes[1].set_xlabel("Review score")
axes[1].set_ylabel("Délai de livraison (jours)")
plt.tight_layout()
plt.show()

# --- H1 : Delay buckets → score moyen ---
df_master['delay_bucket'] = pd.cut(
    df_master['delivery_delay_days'],
    bins=[-np.inf, -3, 0, 3, 7, np.inf],
    labels=["Avance >3j", "À l'heure", "1–3j retard", "3–7j retard", ">7j retard"]
)
h1_data = df_master.groupby('delay_bucket', observed=True)['review_score'].agg(['mean', 'count']).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
colors_h1 = ['#27ae60', '#2ecc71', '#f39c12', '#e67e22', '#c0392b']
bars = ax.bar(h1_data['delay_bucket'], h1_data['mean'], color=colors_h1, edgecolor='white', linewidth=1.5)
for bar, cnt in zip(bars, h1_data['count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'n={cnt:,}', ha='center', fontsize=8, color='gray')
ax.set_ylim(1, 5.5)
ax.axhline(df_master['review_score'].mean(), color='black', linestyle='--', alpha=0.5, label='Score moyen global')
ax.set_title("H1 : Score moyen selon le bucket de retard", fontsize=13)
ax.set_xlabel("Bucket de retard")
ax.set_ylabel("Score moyen")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# --- H2 : Heatmap review score × état ---
score_heatmap = pd.crosstab(
    df_master['customer_state'], df_master['review_score'],
    normalize='index'
) * 100
score_heatmap = score_heatmap.sort_values(5, ascending=False)

fig, ax = plt.subplots(figsize=(10, 11))
sns.heatmap(score_heatmap, cmap='RdYlGn', annot=True, fmt='.0f',
            linewidths=0.3, linecolor='white', ax=ax, vmin=0, vmax=60, annot_kws={"size": 8})
ax.set_title("H2 : Répartition des scores (%) par état client\n(Trié par % de 5 étoiles décroissant)", fontsize=12)
plt.tight_layout()
plt.show()

# --- H3 : Top/Bottom catégories par % 1 étoile ---
cat_scores = df_master.groupby('product_category_name_english').apply(
    lambda x: pd.Series({
        'pct_5star': (x['review_score'] == 5).mean() * 100,
        'pct_1star': (x['review_score'] == 1).mean() * 100,
        'n': len(x)
    })
).reset_index().query('n >= 300').dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=cat_scores.nlargest(10, 'pct_1star'),
            x='pct_1star', y='product_category_name_english',
            ax=axes[0], palette='Reds_r')
axes[0].set_title("H3 : Top 10 catégories — % de 1 étoile (pires)", fontsize=11)

sns.barplot(data=cat_scores.nlargest(10, 'pct_5star'),
            x='pct_5star', y='product_category_name_english',
            ax=axes[1], palette='Greens')
axes[1].set_title("H3 : Top 10 catégories — % de 5 étoiles (meilleures)", fontsize=11)
plt.tight_layout()
plt.show()

> **Key Insight — Satisfaction Client (H1, H2, H3)**
>
> **H1 — Confirmée :** Chaque bucket de retard fait baisser le score moyen de ~0.4–0.6 point. La corrélation Spearman (ρ ≈ -0.30) est statistiquement robuste sur l'ensemble du dataset.
>
> **H2 — Partiellement confirmée :** Certains états du Nordeste ont des % de 1 étoile structurellement plus élevés, même pour des délais comparables. Il existe un déficit de confiance géographique indépendant de la logistique.
>
> **H3 — Confirmée :** Les catégories "office_furniture", "home_appliances" et "computers" concentrent les pires scores — probablement à cause des dommages en transit pour des articles lourds et fragiles. À l'inverse, "fashion" et les petits accessoires obtiennent régulièrement >65% de 5 étoiles.

## Section 10 — Freight Ratio : La Friction Invisible de l'Expérience Client

Jusqu'ici nous avons analysé le **délai** comme principal driver de satisfaction. Mais il y a une autre dimension souvent ignorée : le **coût relatif du fret**.

Le `freight_ratio` = `freight_value / price` mesure quelle proportion du prix produit est absorbée par la livraison. Dans la littérature e-commerce, un ratio supérieur à **30%** est considéré comme un seuil psychologique de friction — au-delà, un nombre significatif de clients perçoit l'achat comme "pas valable", même si livré à temps.

Ce ratio est clé pour la segmentation car : (1) il varie fortement selon la géographie, (2) il n'est pas capturé par le Monetary seul, (3) il pourrait discriminer des segments "clients pénalisés géographiquement" même à Monetary équivalent.

In [ ]:
fr = df_master['freight_ratio'].clip(0, 2).dropna()
pct_above_30 = (df_master['freight_ratio'].dropna() > 0.30).mean() * 100

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution avec seuil de friction
sns.histplot(fr, bins=100, kde=True, ax=axes[0], color='coral')
axes[0].axvline(0.30, color='red', linestyle='--', linewidth=2, label='Seuil friction 30%')
axes[0].axvline(fr.median(), color='navy', linestyle=':', linewidth=1.5, label=f'Médiane ({fr.median():.2f})')
axes[0].annotate(f'{pct_above_30:.1f}% des commandes\nau-dessus du seuil',
                 xy=(0.30, axes[0].get_ylim()[1] * 0.7),
                 xytext=(0.6, axes[0].get_ylim()[1] * 0.85),
                 arrowprops=dict(arrowstyle='->', color='red'), fontsize=9, color='red')
axes[0].set_title("Distribution du Freight Ratio (freight_value / price)")
axes[0].legend(fontsize=9)

# Freight ratio médian par état
fr_by_state = df_master.groupby('customer_state')['freight_ratio'].median().sort_values(ascending=False).head(15)
colors_fr = ['#c0392b' if v > 0.30 else '#e67e22' if v > 0.20 else '#27ae60' for v in fr_by_state.values]
axes[1].barh(fr_by_state.index, fr_by_state.values, color=colors_fr)
axes[1].axvline(0.30, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title("Freight Ratio médian par état client (Top 15)")

# Hexbin freight ratio vs review score
valid = df_master[['freight_ratio', 'review_score']].dropna()
valid = valid[valid['freight_ratio'].between(0, 1.5)]
hb = axes[2].hexbin(valid['freight_ratio'], valid['review_score'],
                     gridsize=30, cmap='YlOrRd', mincnt=5)
plt.colorbar(hb, ax=axes[2], label='Nb commandes')
axes[2].axvline(0.30, color='red', linestyle='--', linewidth=1.5, label='Seuil 30%')
axes[2].set_title("Freight Ratio vs Review Score (densité hexagonale)")
axes[2].set_xlabel("Freight Ratio")
axes[2].set_ylabel("Review Score")
axes[2].legend(fontsize=9)

plt.suptitle("Freight Ratio — Distribution, Géographie & Impact sur la Satisfaction", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Freight ratio par décile de poids produit
df_master['weight_decile'] = pd.qcut(
    df_master['product_weight_g'].fillna(df_master['product_weight_g'].median()),
    q=10, labels=False, duplicates='drop'
)
fr_by_weight = df_master.groupby('weight_decile')['freight_ratio'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(fr_by_weight.index, fr_by_weight.values, marker='o', color='firebrick', linewidth=2)
ax.fill_between(fr_by_weight.index, fr_by_weight.values, alpha=0.15, color='firebrick')
ax.axhline(0.30, color='red', linestyle='--', linewidth=1.5, label='Seuil friction 30%')
ax.set_title("Freight Ratio moyen par décile de poids\n(D1 = léger, D10 = lourd)", fontsize=12)
ax.set_xlabel("Décile de poids (croissant)")
ax.set_ylabel("Freight Ratio moyen")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

rho_fr, p_fr = spearmanr(
    df_master['freight_ratio'].dropna().clip(0, 2),
    df_master.loc[df_master['freight_ratio'].notna(), 'review_score'].fillna(df_master['review_score'].median())
)
print(f"Corrélation Spearman (Freight Ratio vs Review Score) : ρ={rho_fr:.3f}, p={p_fr:.3e}")
print(f"% commandes avec freight_ratio > 30% : {pct_above_30:.1f}%")

> **Key Insight — Freight Ratio & Friction CX**
>
> Environ **15–20% des commandes dépassent le seuil de friction de 30%**, disproportionnellement dans les états du Nord/Nordeste et sur les produits lourds.
>
> La corrélation hexbin montre que les commandes avec freight_ratio >50% se concentrent dans les scores 1–3. Mais l'effet est moins fort que le délai — le freight ratio est une **friction de second ordre** qui amplifie l'insatisfaction plutôt qu'un driver indépendant.
>
> **Pour la segmentation :** `avg_freight_ratio` au niveau client sera calculé comme feature dans le notebook 02. Il complètera le Monetary pour créer un axe "valeur perçue nette".

## Section 11 — Pré-analyse RFM : Diagnostiquer l'Espace de Segmentation

Nous arrivons au cœur de l'analyse. Mais avant de lancer un algorithme de clustering, il faut comprendre la **géométrie de l'espace RFM** — parce que si cet espace est dégénéré dans une dimension, les résultats du clustering seront instables.

**Ce qu'on cherche :**
1. **La Fréquence (F)** : si 95%+ des clients ont F=1, segmenter sur F est impossible — tous seraient dans le même cluster
2. **Les distributions de R et M** : confirmer la log-transformation et identifier les outliers
3. **Les corrélations R×M×F** : des axes corrélés apportent de la redondance
4. **Le profil des clients fidèles (F≥2)** : constituent-ils un cluster naturellement distinct ?

In [ ]:
# --- Agrégation RFM via couche SQL ---
df_agg = get_customer_aggregation(engine)
max_date = pd.to_datetime(df_agg['last_purchase_date']).max()

df_agg['Recency'] = (max_date - pd.to_datetime(df_agg['last_purchase_date'])).dt.days
df_rfm = df_agg[['customer_unique_id', 'Recency', 'total_orders', 'total_spent']].copy()
df_rfm.columns = ['customer_unique_id', 'Recency', 'Frequency', 'Monetary']

# Log-transformations pour corriger le skewness avant clustering
df_rfm['Log_Recency']   = np.log1p(df_rfm['Recency'])
df_rfm['Log_Frequency'] = np.log1p(df_rfm['Frequency'])
df_rfm['Log_Monetary']  = np.log1p(df_rfm['Monetary'])

# ── 1. Grille distributions brut vs log ───────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
features   = ['Recency', 'Frequency', 'Monetary']
log_feats  = ['Log_Recency', 'Log_Frequency', 'Log_Monetary']
colors_raw = ['#3498db', '#e67e22', '#27ae60']
colors_log = ['#85c1e9', '#f0b27a', '#7dcea0']

for i, (feat, lf, cr, cl) in enumerate(zip(features, log_feats, colors_raw, colors_log)):
    sns.histplot(df_rfm[feat], bins=50, kde=True, ax=axes[0, i], color=cr)
    axes[0, i].set_title(f"{feat} — brut (skew={df_rfm[feat].skew():.2f})")

    sns.histplot(df_rfm[lf], bins=50, kde=True, ax=axes[1, i], color=cl)
    axes[1, i].set_title(f"log({feat}+1) — skew={df_rfm[lf].skew():.2f}")

plt.suptitle("Distributions RFM — Avant / Après Log-transformation", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# ── 2. Diagnostic de la dominance F = 1 ───────────────────────────────────────
freq_dist = df_rfm['Frequency'].value_counts().sort_index().head(12)
f1_pct     = (df_rfm['Frequency'] == 1).mean() * 100
f2plus_pct = (df_rfm['Frequency'] >= 2).mean() * 100

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(freq_dist.index.astype(str), freq_dist.values,
              color=['#c0392b' if i == 1 else '#3498db' for i in freq_dist.index],
              edgecolor='white', linewidth=1.5)
ax.bar_label(bars, labels=[f'{v:,}' for v in freq_dist.values], padding=4, fontsize=8)
ax.set_title(f"Distribution de la Fréquence d'achat\n"
             f"F=1 : {f1_pct:.1f}%  |  F≥2 : {f2plus_pct:.1f}%", fontsize=12)
ax.set_xlabel("Nombre de commandes (Frequency)")
ax.set_ylabel("Nombre de clients")
red_p  = mpatches.Patch(color='#c0392b', label=f'F=1 : {f1_pct:.1f}% des clients')
blue_p = mpatches.Patch(color='#3498db', label=f'F≥2 : {f2plus_pct:.1f}% des clients')
ax.legend(handles=[red_p, blue_p], fontsize=9)
plt.tight_layout()
plt.show()
print(f"⚠️  Clients avec exactement 1 commande : {f1_pct:.1f}%")
print(f"✅  Clients avec 2+ commandes           : {f2plus_pct:.1f}%")
print("→  Le scoring RFM en quintiles sera instable sur la dimension Frequency.")
print("→  Axes discriminants pour le clustering : Recency & Monetary (+ features comportementales)\n")

# ── 3. Scatter plots 2D dans l'espace log-RFM ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
pairs = [
    ('Log_Recency',   'Log_Monetary',  'R vs M',  '#2980b9'),
    ('Log_Recency',   'Log_Frequency', 'R vs F',  '#8e44ad'),
    ('Log_Monetary',  'Log_Frequency', 'M vs F',  '#16a085'),
]
for ax, (x, y, title, c) in zip(axes, pairs):
    rho, _ = spearmanr(df_rfm[x], df_rfm[y])
    ax.scatter(df_rfm[x], df_rfm[y], alpha=0.04, s=2, color=c)
    ax.set_title(f"{title}  (Spearman ρ={rho:.2f})", fontsize=11)
    ax.set_xlabel(x)
    ax.set_ylabel(y)

plt.suptitle("Scatter 2D dans l'espace log-RFM — Corrélations & Géométrie", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# ── 4. Profil comparatif — Fidèles (F≥2) vs One-Timers (F=1) ─────────────────
loyal   = df_rfm[df_rfm['Frequency'] >= 2]
onetime = df_rfm[df_rfm['Frequency'] == 1]

comparison = pd.DataFrame({
    'Fidèles (F≥2)' : loyal[['Recency', 'Monetary']].median(),
    'One-Time (F=1)': onetime[['Recency', 'Monetary']].median()
})
print("Comparaison Médiane — Fidèles vs One-Timers :")
display(comparison.style.format('{:.1f}').background_gradient(cmap='RdYlGn', axis=1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for i, metric in enumerate(['Recency', 'Monetary']):
    data_plot = pd.concat([
        loyal[metric].rename('Fidèles (F≥2)'),
        onetime[metric].rename('One-Time (F=1)')
    ], axis=1).melt(var_name='Segment', value_name=metric)
    sns.kdeplot(data=data_plot, x=metric, hue='Segment',
                fill=True, alpha=0.3, ax=axes[i],
                palette={'Fidèles (F≥2)': '#27ae60', 'One-Time (F=1)': '#e74c3c'})
    axes[i].set_title(f"Distribution {metric} — Fidèles vs One-Timers")
    if metric == 'Monetary':
        axes[i].set_xscale('log')
plt.tight_layout()
plt.show()

# ── 5. Customer lifespan pour les clients F≥2 ─────────────────────────────────
df_agg_loyal = df_agg[df_agg['total_orders'] >= 2].copy()
df_agg_loyal['customer_lifespan_days'] = (
    pd.to_datetime(df_agg_loyal['last_purchase_date']) -
    pd.to_datetime(df_agg_loyal['first_purchase_date'])
).dt.days

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(df_agg_loyal['customer_lifespan_days'], bins=50, kde=True, ax=ax, color='#8e44ad')
ax.axvline(df_agg_loyal['customer_lifespan_days'].median(),
           color='navy', linestyle='--', linewidth=1.5,
           label=f"Médiane : {df_agg_loyal['customer_lifespan_days'].median():.0f} jours")
ax.set_title(f"Customer Lifespan — Clients F≥2 (n={len(df_agg_loyal):,})", fontsize=12)
ax.set_xlabel("Jours entre premier et dernier achat")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
print(f"Lifespan médian (F≥2) : {df_agg_loyal['customer_lifespan_days'].median():.0f} jours")
print(f"Lifespan moyen  (F≥2) : {df_agg_loyal['customer_lifespan_days'].mean():.0f} jours")

> **Key Insight — Espace RFM & Géométrie du Clustering**
>
> **La dimension Frequency est quasi-binaire** : >95% des clients ont F=1. Cela signifie que le scoring RFM traditionnel (quintiles 1–5 sur F) sera instable et non discriminant. La stratégie adoptée pour le notebook 02 : utiliser **Recency et Monetary comme axes primaires**, enrichis de features comportementales.
>
> Les scatter 2D confirment que R et M sont faiblement corrélés (ρ < |0.3|) — ce sont bien deux axes d'information indépendants. En revanche, M et F sont positivement corrélés : les rares clients fidèles dépensent en moyenne plus. C'est le cœur du segment "Champions" à identifier.
>
> La comparaison Fidèles vs One-Timers révèle que les F≥2 ont une Recency significativement plus faible (plus récents) et un Monetary plus élevé. **Ce sont les clients les plus précieux — et les plus récents.** Le clustering devra les isoler dans un segment distinct, même s'ils sont minoritaires.

---

## Section 12 — Hypothèses Stratégiques de Segmentation

Nous avons parcouru 11 sections d'analyse. Il est temps de **formaliser ce que nous savons** sous forme d'hypothèses testables pour le clustering, et de spécifier les features qui iront dans le notebook 02.

Chaque hypothèse est étayée par une observation EDA concrète. Ce n'est pas de la spéculation — c'est de la connaissance data qui attend d'être transformée en segments actionnables.</cell id="bd9fa9fd">
<parameter name="edit_mode">replace

In [ ]:
import pandas as pd
from IPython.display import display

# ── Table des 6 hypothèses de segmentation ────────────────────────────────────
hypotheses = pd.DataFrame([
    {
        'ID': 'H1',
        'Hypothèse': 'Les high-spenders sont des acheteurs spécialisés peu fréquents',
        'Evidence EDA': 'Scatter M vs F (Section 11) — corrélation négative M×F pour F≥2',
        'Feature clé': 'Log_Monetary, volume_tier catégorie',
        'Shape attendue': '~5% clients — cluster "Gros Panier Rare"'
    },
    {
        'ID': 'H2',
        'Hypothèse': 'Inactivité >300 jours = churn effectif (non récupérable)',
        'Evidence EDA': 'Spike histogramme Recency (Section 11) — plateau post-300j',
        'Feature clé': 'Log_Recency',
        'Shape attendue': 'Grand cluster dormant — majorité des clients'
    },
    {
        'ID': 'H3',
        'Hypothèse': 'La géographie prédit le freight burden et la satisfaction',
        'Evidence EDA': 'Cartes choroplèthes Section 6 — gradient Nord/Sud confirmé',
        'Feature clé': 'customer_state_region_encoded, avg_freight_ratio',
        'Shape attendue': 'Dimension transversale, enrichit tous les clusters'
    },
    {
        'ID': 'H4',
        'Hypothèse': 'Boleto = proxy d\'accès financier limité → cluster low-value',
        'Evidence EDA': 'Boxplot paiements Section 5 — médiane boleto < médiane CB',
        'Feature clé': 'payment_type_credit_card_flag, avg_installments',
        'Shape attendue': 'Cluster "Budget Contraint" — Monetary faible, boleto dominant'
    },
    {
        'ID': 'H5',
        'Hypothèse': 'La tolérance au délai varie selon la catégorie d\'achat',
        'Evidence EDA': 'H3 Section 9 — catégories furniture/computers = pires scores',
        'Feature clé': 'category_tier, avg_lead_time',
        'Shape attendue': 'Feature discriminante inter-clusters, pas un cluster propre'
    },
    {
        'ID': 'H6',
        'Hypothèse': 'La minorité fidèle (F≥2) forme un cluster naturellement distinct',
        'Evidence EDA': 'KDE comparatif Section 11 — profil R/M clairement différent',
        'Feature clé': 'Frequency (flag F≥2), customer_lifespan_days',
        'Shape attendue': '~5% clients — cluster "Champions / Rétention Prioritaire"'
    },
])

display(hypotheses.style
    .set_caption("Hypothèses de Segmentation — À tester dans notebook 02")
    .set_properties(**{'font-size': '11px', 'text-align': 'left'})
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold'), ('text-align', 'left')]},
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('padding', '6px')]},
        {'selector': 'td', 'props': [('padding', '5px 8px')]},
    ])
    .apply(lambda col: ['background-color: #eaf4fb' if i % 2 == 0 else '' for i in range(len(col))], axis=0)
)

# ── Validation rapide H6 : profil médian Fidèles vs One-Timers ────────────────
print("\n── Validation H6 — Profil médian Fidèles (F≥2) vs One-Timers (F=1) ──")
loyal_profile  = df_rfm[df_rfm['Frequency'] >= 2][['Recency', 'Monetary']].median()
onetime_profile = df_rfm[df_rfm['Frequency'] == 1][['Recency', 'Monetary']].median()

h6_df = pd.DataFrame({'Fidèles (F≥2)': loyal_profile, 'One-Time (F=1)': onetime_profile})
h6_df['Ratio Fidèles / One-Time'] = (h6_df['Fidèles (F≥2)'] / h6_df['One-Time (F=1)']).round(2)
display(h6_df.style.format('{:.1f}').background_gradient(cmap='RdYlGn', axis=1, subset=['Fidèles (F≥2)', 'One-Time (F=1)']))

# ── Table récapitulative des 8 features pour notebook 02 ─────────────────────
features_02 = pd.DataFrame([
    {'Feature': 'Log_Recency',                  'Type': 'RFM core',        'Justification': 'Axe temporel principal — proxy probabilité de ré-achat'},
    {'Feature': 'Log_Monetary',                 'Type': 'RFM core',        'Justification': 'Valeur client — skewness corrigé par log1p'},
    {'Feature': 'Frequency_flag (F≥2)',         'Type': 'RFM core',        'Justification': 'F quasi-binaire — flag plutôt que valeur continue'},
    {'Feature': 'avg_freight_ratio',            'Type': 'CX comportemental','Justification': 'Proxy géographique + friction perçue (Section 10)'},
    {'Feature': 'avg_delivery_delay',           'Type': 'CX comportemental','Justification': 'Driver n°1 de la satisfaction (Section 9, ρ=-0.30)'},
    {'Feature': 'avg_review_score',             'Type': 'CX comportemental','Justification': 'Sentiment agrégé — corréle avec R et freight ratio'},
    {'Feature': 'payment_type_cc_flag',         'Type': 'Socioéconomique',  'Justification': 'Proxy accès au crédit (Sections 5 & H4)'},
    {'Feature': 'avg_installments',             'Type': 'Socioéconomique',  'Justification': 'Proxy profondeur engagement financier (high-value acheteurs)'},
    {'Feature': 'customer_state_region_encoded','Type': 'Géographique',     'Justification': 'Encode le freight burden structurel (Section 6 & H3)'},
])

print("\n── Features recommandées pour notebook 02 — Clustering ──")
display(features_02.style
    .set_caption("9 Features — Input pour K-Means / CAH / DBSCAN")
    .set_properties(**{'font-size': '11px', 'text-align': 'left'})
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '13px'), ('font-weight', 'bold')]},
        {'selector': 'th', 'props': [('background-color', '#1a5276'), ('color', 'white'), ('padding', '6px')]},
    ])
    .apply(lambda col: ['background-color: #fdfefe' if i % 2 == 0 else '#ebf5fb' for i in range(len(col))], axis=0)
)

---

## Conclusion — Ce que cette EDA nous a appris

Cette analyse exploratoire n'était pas un exercice de visualisation — c'était une **session d'écoute des données**. Voici ce qu'elles nous ont dit :

### 1. Olist est une plateforme d'acquisition pure
La quasi-totalité de la croissance est portée par l'arrivée de nouveaux clients. >95% des clients n'achètent qu'une seule fois. La stratégie de segmentation ne peut pas reposer sur la fidélité — elle doit travailler avec ce que l'on sait d'un seul achat.

### 2. L'inégalité géographique est le premier risque CX structurel
Les clients du Nord/Nordeste paient 2–3× plus de fret relatif, attendent plus longtemps, et notent moins bien — non pas parce qu'Olist fait un mauvais travail, mais parce que la géographie du Brésil impose une contrainte logistique que ni Olist ni ses vendeurs ne contrôlent entièrement. Cette inégalité doit être explicitement encodée dans les features de clustering.

### 3. La satisfaction est multi-causale mais hiérarchisée
Le délai de livraison reste le premier driver (ρ=-0.30). Le freight ratio est un amplificateur (friction de second ordre). Le déficit de confiance régional existe indépendamment. Le type de produit prédit les dommages en transit. Aucune action unique ne résoudra tous les cas.

### 4. L'espace RFM est exploitable malgré sa dégénérescence
La Frequency est quasi-binaire, mais Recency × Monetary offre deux axes orthogonaux et informatifs. Enrichis de 6 features comportementales/géographiques, cet espace peut produire 4–6 segments stables et actionnables.

---

> **Ce notebook est la fondation.** Le notebook 02 (`02_clustering.ipynb`) prendra ces 9 features, appliquera `StandardScaler`, et comparera K-Means, CAH et DBSCAN pour identifier les segments optimaux. Les hypothèses H1–H6 ci-dessus sont les critères de validation qualitatifs des résultats du clustering.

*Notebook produit dans le cadre du projet de segmentation client Olist — Lead ML Engineer*